# ⚡ Quantum ML Fault Detection
### Hybrid Quantum-Classical Model for Electrical Grid Stability

**Dataset:** Electrical Grid Stability Simulated Data  
**Task:** Binary classification — predict if the grid is `stable` or `unstable`  
**Approach:** Hybrid model — Classical FC layers → Quantum circuit (PennyLane) → Classical output layer

---
### Fixes & Improvements over v1:
- ✅ Fixed data leakage (removed `stab` numeric column from features)
- ✅ Fixed output layer size (2 classes, not 3)
- ✅ Fixed `AngleEmbedding` input range (scaled to [-π, π])
- ✅ Used all 12 features with 12 qubits
- ✅ Added `random_state` for reproducibility
- ✅ Added mini-batch training
- ✅ Added classical baseline (Random Forest) for comparison
- ✅ Added loss/accuracy curves
- ✅ Added confusion matrix and classification report


## 1. Install Dependencies

In [ ]:
!pip install pennylane -q

## 2. Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
# If running in Colab, upload your sg_raw_data.csv first
# from google.colab import files
# uploaded = files.upload()

data = pd.read_csv("sg_raw_data.csv")

print("Shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())
print("\nClass distribution:")
print(data['stabf'].value_counts())
print("\nMissing values:", data.isnull().sum().sum())

## 3. Preprocess Data

> **Bug fix:** The original code kept the `stab` column (a numeric version of the target `stabf`). This is **data leakage** — the model was essentially seeing the answer. We now drop both `stab` and `stabf` from features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# FIX: Drop BOTH 'stabf' (string label) AND 'stab' (numeric label — was data leakage!)
X = data.drop(["stabf", "stab"], axis=1).values   # 12 features: tau1-4, p1-4, g1-4
y = data["stabf"].values

# Encode labels: stable=1, unstable=0
le = LabelEncoder()
y = le.fit_transform(y)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Feature matrix shape:", X.shape)

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# FIX: Added random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 4. Classical Baseline (Random Forest)

Before building the quantum model, we establish a **classical baseline**. This is essential — it lets us honestly measure whether the quantum model adds value.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print(f"Random Forest Accuracy: {rf_acc * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, rf_preds, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 5. Quantum Circuit

> **Fix:** Original used 6 qubits but had 12 features — `AngleEmbedding` was silently dropping 6 features.  
> **Fix:** Inputs are now scaled to `[-π, π]` before embedding, which is the correct range for angle-based encoding.

In [ ]:
import pennylane as qml
from pennylane import numpy as pnp

# FIX: Use 12 qubits to match all 12 features
n_qubits = 12
n_layers = 2   # Number of StronglyEntanglingLayers

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    # FIX: Scale inputs to [-pi, pi] for proper angle encoding
    scaled_inputs = inputs * pnp.pi
    qml.AngleEmbedding(scaled_inputs, wires=range(n_qubits), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Preview circuit structure
print("Quantum circuit preview:")
dummy_inputs = pnp.zeros(n_qubits)
dummy_weights = pnp.zeros((n_layers, n_qubits, 3))
print(qml.draw(quantum_circuit)(dummy_inputs, dummy_weights))

## 6. Hybrid Model

> **Fix:** Output layer now correctly uses 2 classes (binary: stable / unstable), not 3.

In [ ]:
import torch
import torch.nn as nn

# Build the quantum layer
weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

class HybridModel(nn.Module):
    """
    Hybrid Quantum-Classical Neural Network for fault detection.

    Architecture:
        Input (12) -> FC(32) -> ReLU -> Dropout
                   -> FC(12) -> Quantum Circuit (12 qubits)
                   -> FC(8)  -> ReLU
                   -> FC(2)  -> Output (stable / unstable)
    """
    def __init__(self, input_size):
        super().__init__()
        self.pre_net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, n_qubits),
            nn.Tanh()   # Tanh keeps values in [-1,1], good before angle embedding
        )
        self.q_layer = qlayer
        self.post_net = nn.Sequential(
            nn.Linear(n_qubits, 8),
            nn.ReLU(),
            nn.Linear(8, 2)  # FIX: 2 output classes, not 3
        )

    def forward(self, x):
        x = self.pre_net(x)
        x = self.q_layer(x)
        x = self.post_net(x)
        return x


# Set seed for reproducibility
torch.manual_seed(42)
model = HybridModel(X_train.shape[1])

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture:\n{model}")
print(f"\nTotal trainable parameters: {total_params}")

## 7. Training with Mini-Batches

> **Improvement:** Original used full-batch gradient descent. Mini-batch training generalizes better and is the standard approach.

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# DataLoader for mini-batch training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# ---- Training loop ----
epochs = 60
train_losses, train_accs, test_accs = [], [], []

print(f"Training for {epochs} epochs...\n")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * len(X_batch)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)

    scheduler.step()

    avg_loss = epoch_loss / total
    train_acc = correct / total

    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_t)
        test_preds = test_outputs.argmax(dim=1)
        test_acc = (test_preds == y_test_t).float().mean().item()

    train_losses.append(avg_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{epochs}] "
              f"Loss: {avg_loss:.4f} | "
              f"Train Acc: {train_acc*100:.2f}% | "
              f"Test Acc: {test_acc*100:.2f}%")

print("\nTraining complete!")

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(train_losses, color='steelblue', linewidth=2)
ax1.set_title('Training Loss', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(train_accs, label='Train', color='steelblue', linewidth=2)
ax2.plot(test_accs,  label='Test',  color='coral',     linewidth=2, linestyle='--')
ax2.set_title('Accuracy', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.suptitle('Hybrid Quantum-Classical Model — Training History', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Evaluation & Confusion Matrix

In [ ]:
model.eval()
with torch.no_grad():
    final_preds = model(X_test_t).argmax(dim=1).numpy()

qml_acc = accuracy_score(y_test, final_preds)

print("=" * 50)
print(f"  Quantum Model Test Accuracy : {qml_acc * 100:.2f}%")
print(f"  Random Forest Accuracy      : {rf_acc * 100:.2f}%")
print("=" * 50)

print("\nQuantum Model — Classification Report:")
print(classification_report(y_test, final_preds, target_names=le.classes_))

# Confusion matrix
cm_q = confusion_matrix(y_test, final_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_q, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Quantum Model — Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 10. Model Comparison Summary

In [ ]:
models = ['Random Forest\n(Classical)', 'Hybrid Quantum-\nClassical']
accs   = [rf_acc * 100, qml_acc * 100]
colors = ['#4A90D9', '#E8834A']

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accs, color=colors, width=0.45, edgecolor='white', linewidth=1.5)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
plt.ylim(0, 110)
plt.ylabel('Test Accuracy (%)', fontsize=11)
plt.title('Classical vs Quantum Model Comparison', fontsize=13)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📋 Summary of all fixes and improvements:")
print("-" * 55)
improvements = [
    ("✅ Fixed",  "Dropped 'stab' column — was data leakage"),
    ("✅ Fixed",  "Output layer = 2 classes (was 3)"),
    ("✅ Fixed",  "AngleEmbedding inputs scaled to [-π, π]"),
    ("✅ Fixed",  "Used all 12 features with 12 qubits"),
    ("✅ Fixed",  "Added random_state=42 for reproducibility"),
    ("🚀 Added",  "Mini-batch training with DataLoader"),
    ("🚀 Added",  "LR scheduler (StepLR)"),
    ("🚀 Added",  "Dropout in pre-network"),
    ("🚀 Added",  "Classical baseline (Random Forest)"),
    ("🚀 Added",  "Loss & accuracy training curves"),
    ("🚀 Added",  "Confusion matrix & classification report"),
    ("🚀 Added",  "Model comparison bar chart"),
]
for tag, desc in improvements:
    print(f"  {tag:10s} {desc}")